# 🧪 ML Experiment Tracker – Quant Researcher
**Workflow**: Feature Load → Train model → Log to MLflow → Register model → Compare runs

---

In [ ]:
from qtrader.analyst import AnalystSession, RoleContext
from qtrader.features.store import FeatureStore
import polars as pl
import numpy as np

session = AnalystSession(role=RoleContext.RESEARCHER)

SYMBOL   = 'BTC-USD'
TIMEFRAME = '1d'
TARGET   = 'fwd_ret_1'
EXPERIMENT = 'qtrader_alpha_research'
TEST_FRAC = 0.2

## 1. Load Features from FeatureStore

In [ ]:
df = session.load_features(SYMBOL, TIMEFRAME)

if df.is_empty():
    print("⚠️  No stored features. Generating on-the-fly...")
    df = session.sample_ohlcv(symbol='BTC', days=365)
    df = session.make_returns(df)
    df = session.add_rolling_features(df)
    df = session.run_alpha_score(df, forward_periods=[1, 5])
    df = df.drop_nulls()

feature_cols = [c for c in df.columns if c.startswith(('sma_', 'vol_', 'rsi_', 'avg_'))]
print(f"Features: {feature_cols}")
print(f"Target: {TARGET}")
df.head(3)

## 2. Train/Test Split

In [ ]:
valid_df = df.select(feature_cols + [TARGET]).drop_nulls()
n = len(valid_df)
split = int(n * (1 - TEST_FRAC))
train_df = valid_df.head(split)
test_df  = valid_df.tail(n - split)

X_train = train_df.select(feature_cols).to_numpy()
y_train = train_df[TARGET].to_numpy()
X_test  = test_df.select(feature_cols).to_numpy()
y_test  = test_df[TARGET].to_numpy()
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

## 3. MLflow Experiment: Train & Log

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

mlflow.set_experiment(EXPERIMENT)

params = {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 3, 'subsample': 0.8}

with mlflow.start_run(run_name=f'{SYMBOL}_GBR') as run:
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_train)
    X_te_sc = scaler.transform(X_test)

    model = GradientBoostingRegressor(**params, random_state=42)
    model.fit(X_tr_sc, y_train)

    train_mse = mean_squared_error(y_train, model.predict(X_tr_sc))
    test_mse  = mean_squared_error(y_test,  model.predict(X_te_sc))
    ic = float(np.corrcoef(model.predict(X_te_sc), y_test)[0, 1])

    mlflow.log_params(params)
    mlflow.log_metrics({'train_mse': train_mse, 'test_mse': test_mse, 'IC': ic})
    mlflow.sklearn.log_model(model, artifact_path='model')

    print(f"Run ID: {run.info.run_id}")
    print(f"Train MSE: {train_mse:.6f} | Test MSE: {test_mse:.6f} | IC: {ic:.4f}")

## 4. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

fi = model.feature_importances_
idx = np.argsort(fi)[::-1]
feat_names = np.array(feature_cols)[idx]
fi_sorted  = fi[idx]

fig, ax = plt.subplots(figsize=(10, 4), facecolor='#0f1117')
ax.set_facecolor('#0f1117')
for sp in ax.spines.values(): sp.set_color('#334155')
ax.tick_params(colors='#94a3b8')
ax.grid(color='#1e293b', linestyle='--', linewidth=0.5, axis='x')
bars = ax.barh(feat_names, fi_sorted, color='#38bdf8')
ax.set_title('Feature Importance (GBR)', color='#e2e8f0')
ax.set_xlabel('Importance', color='#94a3b8')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Compare All Runs in Experiment

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=['metrics.IC DESC'])
if not runs.empty:
    display_cols = ['run_id', 'params.n_estimators', 'params.learning_rate',
                    'metrics.train_mse', 'metrics.test_mse', 'metrics.IC']
    available = [c for c in display_cols if c in runs.columns]
    print(runs[available].head(10).to_string(index=False))